In [1]:
# libraries
import os
import json
import time
import subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import nbformat
from IPython.display import display

In [2]:
# globals; run through run_pipeline.ipynb
ROOT = Path(os.environ['NCA_PROJECT_ROOT'])
TASK = ROOT / 'f_phase2_train'
INPUT, OUTPUT = TASK / 'input', TASK / 'output'
CFG = json.loads((INPUT / 'config.json').read_text())
DEVICE = torch.device(CFG['device'])
torch.set_num_threads(CFG['cpu_threads'])
torch.use_deterministic_algorithms(True)
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = False
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False

In [3]:
# functions passed through c_models/output; definitions only, no training side effects
runtime_nb = nbformat.read(INPUT / 'runtime.ipynb', as_version=4)
for runtime_cell in runtime_nb.cells:
    exec(compile(runtime_cell.source, 'runtime.ipynb', 'exec'), globals())
PACK = torch.load(INPUT / 'dataset.pt', weights_only=True, map_location='cpu')
SPECS = CFG['models']

In [4]:
# content identities for exact checkpoint resumption
data_hash = hashlib.sha256((INPUT / 'dataset.pt').read_bytes()).hexdigest()
code_hash = (INPUT / 'runtime.sha256').read_text()
run_rows = []

In [5]:
# Phase 1 best checkpoints are initialization only; Phase 2 gets its own optimizer and selection
phase1_manifest = pd.read_csv(INPUT / 'phase1_manifest.csv')

In [6]:
# train Phase 2; identical condition/augmentation schedules within each paired replicate
runs = OUTPUT / 'phase2_runs'
runs.mkdir(exist_ok=True)
for regime in CFG['budget_regimes']:
    for replicate in range(CFG['replicates']):
        for spec in SPECS:
            name = spec['name']
            run_name = f'{regime}__{name}__r{replicate}'
            phase1_row = phase1_manifest.query('model == @name and regime == @regime and replicate == @replicate').iloc[0]
            warm_start = INPUT / 'phase1_runs' / phase1_row['checkpoint'] if CFG['phase2_initialization'] == 'phase1' else None
            result = fit_run(spec, CFG, PACK, 2, regime, replicate, runs / run_name,
                             data_hash, code_hash, warm_start=warm_start)
            run_rows.append(result)
            pd.DataFrame(run_rows).to_csv(OUTPUT / 'phase2_manifest.csv', index=False)
display(pd.DataFrame(run_rows))

,step,train_loss,val_loss,grad_norm,switches_this_batch,total_bits,total_cell_updates,seconds,phase,model,regime,replicate
0,2,0.140481,0.118727,0.22699,0,86480,11520,0.047773,2,1A_det,updates,0


,step,train_loss,val_loss,grad_norm,switches_this_batch,total_bits,total_cell_updates,seconds,phase,model,regime,replicate
0,4,0.148706,0.114145,0.231929,0,172960,23040,0.091096,2,1A_det,updates,0


,step,train_loss,val_loss,grad_norm,switches_this_batch,total_bits,total_cell_updates,seconds,phase,model,regime,replicate
0,2,0.143352,0.123323,0.17481,0,86480,11520,0.0345,2,1A_bern,updates,0


,step,train_loss,val_loss,grad_norm,switches_this_batch,total_bits,total_cell_updates,seconds,phase,model,regime,replicate
0,4,0.153039,0.120026,0.17995,0,172960,23040,0.073358,2,1A_bern,updates,0


,step,train_loss,val_loss,grad_norm,switches_this_batch,total_bits,total_cell_updates,seconds,phase,model,regime,replicate
0,2,0.146875,0.128817,0.135114,0,172960,11520,0.057547,2,1B_det,updates,0


,step,train_loss,val_loss,grad_norm,switches_this_batch,total_bits,total_cell_updates,seconds,phase,model,regime,replicate
0,4,0.158386,0.12754,0.139412,0,345920,23040,0.123357,2,1B_det,updates,0


... (15 outputs left out)

,step,train_loss,val_loss,grad_norm,switches_this_batch,total_bits,total_cell_updates,seconds,phase,model,regime,replicate
0,4,0.159842,0.129715,0.074603,0,172960,11520,0.067558,2,1B_rbern,bits,0


,step,train_loss,val_loss,grad_norm,switches_this_batch,total_bits,total_cell_updates,seconds,phase,model,regime,replicate
0,2,0.147429,0.129648,0.079512,0,86480,5760,0.029903,2,1B_qrbern,bits,0


,step,train_loss,val_loss,grad_norm,switches_this_batch,total_bits,total_cell_updates,seconds,phase,model,regime,replicate
0,4,0.159015,0.128417,0.084522,0,172960,11520,0.067106,2,1B_qrbern,bits,0


,phase,model,family,regime,replicate,parameters,width,best_step,best_val_loss,signature,optimizer_steps,training_bits,training_cell_updates,seconds,checkpoint
0,2,1A_det,message,updates,0,1219,22,4,0.114145,05c2aad9796413c7ceab3249ab1884a8b55c850c7deead...,4,172960,23040,0.097945,updates__1A_det__r0/best.pt
1,2,1A_bern,message,updates,0,1219,22,4,0.120026,f7ddaf76b4b7c287ad60181affa056f7005a88fd9c5b54...,4,172960,23040,0.079775,updates__1A_bern__r0/best.pt
2,2,1B_det,query_answer,updates,0,1160,10,4,0.127540,6bc04c2124d8c68ae3bb4a640c9fb0cf492b5cd14c2385...,4,345920,23040,0.130726,updates__1B_det__r0/best.pt
3,2,1B_qbern,query_answer,updates,0,1160,10,4,0.125595,0f8ead0a78b1f7ea65d03503fc1e3f0c8d3a8047933024...,4,345920,23040,0.139197,updates__1B_qbern__r0/best.pt
4,2,1B_rbern,query_answer,updates,0,1160,10,4,0.126834,2a8083f9c281462bf014cdc949042e6ec4ebff3b4eba1a...,4,345920,23040,0.133195,updates__1B_rbern__r0/best.pt
5,2,1B_qrbern,query_answer,updates,0,1160,10,4,0.124391,56214ea3ec87087fc0a9f565671e56ca5a82b6d45e7675...,4,345920,23040,0.130779,updates__1B_qrbern__r0/best.pt
6,2,1A_det,message,bits,0,1219,22,4,0.114145,dc624cf2896862ed1ddba05599546ec7f5474331fab118...,4,172960,23040,0.079462,bits__1A_det__r0/best.pt
7,2,1A_bern,message,bits,0,1219,22,4,0.120026,d04ba6e2788cd7e81c8b2bf9f42c7a4735a5dbd80a4c2a...,4,172960,23040,0.084511,bits__1A_bern__r0/best.pt
8,2,1B_det,query_answer,bits,0,1160,10,4,0.130053,d7c36a09253f9bc17b2c54cf7c868f4868b16d2d4beefa...,4,172960,11520,0.074652,bits__1B_det__r0/best.pt
9,2,1B_qbern,query_answer,bits,0,1160,10,4,0.129025,e2e6e09a4f4ca3539ff6d66cd265db7914318b79a4c1cc...,4,172960,11520,0.077114,bits__1B_qbern__r0/best.pt
